# Import

In [1]:
import string
import IPython
from IPython.display import Audio
import torch
import os

import torchaudio

from TTS.tts.utils.synthesis import synthesis
from TTS.utils.audio import AudioProcessor
from TTS.tts.models import setup_model
from TTS.config import load_config
from TTS.tts.models.vits import *
from TTS.tts.utils.speakers import SpeakerManager
from TTS.utils.vad import get_vad_model_and_utils, remove_silence, resample_wav, read_audio

from pydub import AudioSegment

# Define Parameter and Constant

In [2]:
OUT_PATH = 'output'
BASE_MODEL_PATH = './model/th_en-denoised_lj'
REFERENCE_FILENAME = 'Tsync2_905_mic1.flac'

# model vars 
MODEL_PATH = os.path.join(BASE_MODEL_PATH, 'best_model_76826.pth')
CONFIG_PATH = os.path.join(BASE_MODEL_PATH, 'config.json')
TTS_LANGUAGES = os.path.join(BASE_MODEL_PATH, 'language_ids.json')
USE_CUDA = torch.cuda.is_available()
REFERENCE_PATH = os.path.join("./reference_voice", REFERENCE_FILENAME.strip().strip('./').strip('/'))
REFERENCE_WAV_PATH = os.path.join("./reference_voice", "wav", REFERENCE_FILENAME.strip().strip('./').strip('/').split(".")[0] + ".wav")
SPEAKER_FOLDER = REFERENCE_PATH.split("/")[-1].split(".")[0]

model_name = MODEL_PATH.rstrip('.pth').split('/')[-1]

# Setup Model and Config

In [ ]:
# load the config
C = load_config(CONFIG_PATH)

# load the audio processor
ap = AudioProcessor(**C.audio)

# override config
C["speakers_file"] = None
C["d_vector_file"] = []
C["language_ids_file"] = TTS_LANGUAGES

C["model_args"]["speakers_file"] = None
C["model_args"]["d_vector_file"] = []
C["model_args"]["language_ids_file"] = TTS_LANGUAGES

C.model_args['use_speaker_encoder_as_loss'] = False

model = setup_model(C)
cp = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

# remove speaker encoder
model_weights = cp['model'].copy()
for key in list(model_weights.keys()):
  if "speaker_encoder" in key:
    del model_weights[key]

model.load_state_dict(model_weights)

model.eval()

if USE_CUDA:
  model = model.cuda()

# synthesize voice
use_griffin_lim = False

# Process reference audio file

In [55]:
# Check if refernec voice is in wav format
current_ref_extension = REFERENCE_PATH.split(".")[-1].lower()

# Convert to wav if not or never converted
if current_ref_extension != "wav":
    if not os.path.exists(REFERENCE_WAV_PATH):
        audio: AudioSegment = AudioSegment.from_file(REFERENCE_PATH, format=current_ref_extension)
        audio.export(REFERENCE_WAV_PATH, format="wav")
    REFERENCE_PATH = REFERENCE_WAV_PATH

In [ ]:
# Resmapling reference voice if needed
ref_wav, current_sr = read_audio(REFERENCE_PATH)
if current_sr != C.audio['sample_rate']:
    print('Resampling reference audio...')
    resamapled_ref_wav = resample_wav(ref_wav, current_sr, C.audio['sample_rate'])
    torchaudio.save(REFERENCE_PATH, resamapled_ref_wav[None, :], C.audio['sample_rate'])
else:
    print('This audio is already in the correct sample rate')

In [ ]:
# trim silence at the beginning and end of the audio
model_and_utils = get_vad_model_and_utils(use_cuda=USE_CUDA, use_onnx=False)

output_path, is_speech = remove_silence(
  model_and_utils,
  REFERENCE_PATH,
  REFERENCE_PATH,
  trim_just_beginning_and_end=True,
  use_cuda=USE_CUDA
)

In [58]:
# normalize the reference audio with rms to -27dB
!ffmpeg-normalize $REFERENCE_PATH -nt rms -t=-27 -o $REFERENCE_PATH -ar 16000 -f

In [ ]:
SE_speaker_manager = SpeakerManager(encoder_model_path=C["model_args"]["speaker_encoder_model_path"], encoder_config_path=C["model_args"]["speaker_encoder_config_path"], use_cuda=USE_CUDA)
reference_emb = SE_speaker_manager.compute_embedding_from_clip(REFERENCE_PATH)

# Setup duration predictor

In [60]:
model.length_scale = 1 # scaler for the duration predictor. The larger it is, the slower the speech.
model.inference_noise_scale = 0.2 # defines the noise variance applied to the random z vector at inference.
model.inference_noise_scale_dp = 0.8 # defines the noise variance applied to the duration predictor z vector at inference.

# Setup Language

In [ ]:
# Select language
language_id = 1
language_name_to_id = model.language_manager.name_to_id
language_id_to_name = {v: k for k, v in language_name_to_id.items()}
print(f"Language ID: {language_id}, Language Name: {language_id_to_name[language_id]}")

# Inference

In [ ]:
text = "ยินดีที่ได้รู้จักครับ ผมชื่อภีม วันนี้ผมมาขายเฉาก๊วยชากังราว. am way ใช้เทคโนโลยี technology ozone โอโซนนี้ไม่ฆ่าแมวนะ และ reverse osmosis รีเวิสออสโมซิส."
print(f" > text: {text} with sampling rate: {ap.sample_rate}")

wav, alignment, _, _ = synthesis(
                    model = model,
                    text = text,
                    CONFIG = C,
                    use_cuda = USE_CUDA,
                    d_vector = reference_emb,
                    style_wav = None,
                    language_id = language_id,
                    use_griffin_lim = True,
                    do_trim_silence = False,
                ).values()
print("Audio Generated")
IPython.display.display(Audio(wav, rate=ap.sample_rate))

# Save to Folder

In [ ]:
file_name = text.replace(" ", "_")
file_name = model_name + '_' + file_name.translate(str.maketrans('', '', string.punctuation.replace('_', ''))) + '.wav'
out_path = os.path.join(OUT_PATH, f"{SPEAKER_FOLDER}/{file_name}")

print(f" > Saving output to {out_path}")

os.makedirs(f"{OUT_PATH}/{SPEAKER_FOLDER}", exist_ok=True)
ap.save_wav(wav, out_path)